<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/flagships/humanoid-lab/lessons/F15-L02-kinematics-jacobians/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/flagships/humanoid-lab/lessons/F15-L02-kinematics-jacobians/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=flagships/humanoid-lab/lessons/F15-L02-kinematics-jacobians/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have. The cell below installs `mujoco==3.13.0` and fetches the
files it needs beside it, and does nothing where they are already present. On Kaggle, switch
Internet on in the notebook's settings first; Kaggle allows that only for phone-verified
accounts.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = [("mujoco", "mujoco==3.13.0")]            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = ["assets/SOURCE.md", "assets/arm.xml"]    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/flagships/humanoid-lab/lessons/F15-L02-kinematics-jacobians/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# F15-L02 · Kinematics, frames and Jacobians

**You will build:** a forward-kinematics lookup for any named body or site, the 3-by-nv
translational Jacobian of a humanoid palm, a finite-difference check that proves your
Jacobian is the derivative it claims to be, and a damped least-squares inverse-kinematics
solver that walks the hand onto a Cartesian target and reports the residual.

**Time:** ~50 minutes · **Runs on:** a laptop CPU, no GPU, no download
· **Prerequisites:** F15-L01 (mjModel vs mjData, nq vs nv, forward kinematics as a stage)

By the end you will be able to:
1. Return a named body's and a named site's world position for any `qpos`, and measure how
   far apart the body frame origin, the body centre of mass and a site on it really are.
2. Build the translational Jacobian with `mj_jac`, and show that the two dofs which cannot
   move the hand have columns of exactly zero.
3. Verify that analytic Jacobian against central finite differences of your own forward
   kinematics, to a stated tolerance.
4. Measure how that agreement degrades at both large and small step sizes, and find the
   best step size from your own sweep rather than from folklore.
5. Implement damped least-squares inverse kinematics, report the residual for a reachable
   and an unreachable target, and measure what the damping term prevents.

Every number in this notebook's output is computed by the code you run. Nothing numeric is
typed by hand; the sources for what MuJoCo *documents* are in `claims.yaml`.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import hashlib
import io
import sys
from pathlib import Path
from typing import Callable

import mujoco
import numpy as np

np.set_printoptions(precision=5, suppress=True, linewidth=110)
print("mujoco", mujoco.__version__, "· numpy", np.__version__)

MODEL_FILENAME = "arm.xml"


def arm_xml_path() -> Path:
    """Locate the arm model that ships next to this notebook.

    There is no download branch. The model was written for this lesson and lives in
    `assets/`; if it is missing, the checkout is broken and saying so beats a silent fetch.
    """
    try:
        here = Path(__file__).resolve().parent
    except NameError:  # a notebook has no __file__
        here = Path.cwd()
    for candidate in (here / "assets" / MODEL_FILENAME,
                      here.parent / "assets" / MODEL_FILENAME,
                      Path.cwd() / "assets" / MODEL_FILENAME,
                      Path.cwd().parent / "assets" / MODEL_FILENAME):
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"{MODEL_FILENAME} not found next to this lesson. It ships in assets/ and is never "
        "downloaded; restore it from the lesson directory."
    )


def load_arm():
    """Compile the arm and hand back a fresh (model, data) pair."""
    model = mujoco.MjModel.from_xml_path(str(arm_xml_path()))
    return model, mujoco.MjData(model)


MODEL, DATA = load_arm()

HAND_BODY = "hand_right"
PALM_SITE = "palm"

# The eight dofs that can move the right palm, and the two that cannot. Lists, not tuples:
# a tuple used as a NumPy index means something else entirely.
ARM_DOFS = [0, 1, 2, 3, 4, 5, 6, 7]
LEFT_ARM_DOFS = [8, 9]

# Two poses, both read from the model rather than typed. HOME is qpos = 0, where the right
# arm hangs straight down, fully extended — you will see in section 11 why that matters.
HOME = np.zeros(MODEL.nq)
_key = mujoco.MjData(MODEL)
mujoco.mj_resetDataKeyframe(MODEL, _key, 0)
READY = _key.qpos.copy()

# Joint travel limits, straight off the compiled model. All ten joints are hinges, so row i
# of jnt_range is the limit of dof i.
JOINT_LO = MODEL.jnt_range[:, 0].copy()
JOINT_HI = MODEL.jnt_range[:, 1].copy()

FD_EPS = 1e-6      # the default step. Section 8 measures where the error bottoms out.
JAC_TOL = 1e-6     # the stated tolerance your Jacobian must meet. Section 8 says why.

print(f"model: nq={MODEL.nq} nv={MODEL.nv} nu={MODEL.nu} "
      f"nbody={MODEL.nbody} nsite={MODEL.nsite}")
print("joints, in dof order:")
for _j in range(MODEL.njnt):
    _name = mujoco.mj_id2name(MODEL, mujoco.mjtObj.mjOBJ_JOINT, _j)
    print(f"  dof {int(MODEL.jnt_dofadr[_j]):2d}  qpos {int(MODEL.jnt_qposadr[_j]):2d}  "
          f"{_name:<18s} range [{MODEL.jnt_range[_j][0]:+.2f}, {MODEL.jnt_range[_j][1]:+.2f}]")
print(f"nq - nv = {MODEL.nq - MODEL.nv}  (every joint is a hinge, so dof i IS qpos slot i)")

# Run all is safe before you have written a line: every check, and every demo that needs your
# code, goes through _try, which reports and carries on. The board at the foot of the notebook
# shows where you stand.
_EXERCISES = {"exercise 1": "body_world_position", "exercise 2": "site_world_position",
              "exercise 3": "analytic_jacobian", "exercise 4": "finite_difference_jacobian",
              "exercise 5": "ik_damped_least_squares", "self-check": "your four letters"}
_STATUS: dict = {}      # label -> "passed" | "failed" | "not started", read by the board


def _try(label: str, check: Callable[[], None], needs: tuple = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. Anything listed in `needs` must have
    passed first; until it has, this names it and skips rather than failing on its behalf.
    Nothing is swallowed: every outcome lands in _STATUS, and the `__main__` block at the foot
    of this file exits non-zero outside a notebook if any check failed.
    """
    waiting = [n for n in needs if _STATUS.get(n) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        named = " and ".join(f"{n} ({_EXERCISES.get(n, n)})" for n in waiting)
        one = len(waiting) == 1
        print(f"{label}: skipped until {named} {'passes' if one else 'pass'} — finish "
              f"{'that' if one else 'those'}, then re-run this cell.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        print(f"{label}: {exc}" if str(exc) else
              f"{label}: not implemented yet — fill in the stub above, then re-run this cell.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. Forward kinematics is a stage, not a function you call by accident

`qpos` is the input. Everything you actually want to *read* — where the hand is, where the
palm is, which way the elbow axis points — is derived from it, and none of it updates when
you write to `qpos`. MuJoCo's computation chapter puts forward kinematics first in the
position stage, before anything else; until that stage runs again, you are reading the
previous pose.

Run this. The first printed position is stale, and the notebook proves it rather than
telling you.

In [ ]:
_m, _d = load_arm()
_sid = mujoco.mj_name2id(_m, mujoco.mjtObj.mjOBJ_SITE, PALM_SITE)
mujoco.mj_forward(_m, _d)
print(f"palm at qpos=0                 : {_d.site_xpos[_sid]}")
_before = np.array(_d.site_xpos[_sid])               # a COPY, so it survives the refresh below
_d.qpos[:] = READY                                   # the write, with no refresh
print(f"after writing a new qpos       : {_d.site_xpos[_sid]}   <- unchanged, and wrong")
mujoco.mj_forward(_m, _d)                            # the refresh
print(f"after mujoco.mj_forward        : {_d.site_xpos[_sid]}   <- now it means something")
_moved = float(np.linalg.norm(_d.site_xpos[_sid] - _before))
print(f"the write moved it by {_moved:.3f} m, none of which you could see until the stage ran")

## 2. Exercise 1 — `body_world_position`

The first instrument: given a pose, where is a named body? Three things to get right, and
each of them is a bug somebody has shipped.

- **Look the name up**, do not hard-code an index. `mujoco.mj_name2id(model,
  mujoco.mjtObj.mjOBJ_BODY, name)` returns -1 for a name that does not exist, and -1 is a
  perfectly valid Python index that will hand you the last body without complaining.
- **Refresh the position stage** after writing `qpos`, exactly as section 1 showed.
- **Copy the answer out.** `data.xpos[i]` is a live view into `mjData`. Return it and the
  next call to this function silently rewrites the answer you already have.

<details><summary>💡 Hint 1 — what to think about</summary>

Each bullet above is a separate way to return a plausible vector that is wrong. Which array
is the frame origin rather than the centre of mass? What makes a write to `qpos` show up in
anything derived from it? And what becomes of an array you returned once the next call runs
the position stage again?

</details>

<details><summary>💡 Hint 2 — the approach</summary>

Look the id up by name under the body object type. Only if a `qpos` was passed, write it
into `data.qpos`; either way, run `mujoco.mj_forward` next. Then read that body's row of
`data.xpos` and hand back a fresh array built from it, never the row itself.

</details>

In [ ]:
def body_world_position(model, data, body_name: str, qpos=None) -> np.ndarray:
    """Return the world position of a named body's FRAME ORIGIN, as a length-3 array.

    If `qpos` is given, write it into `data.qpos` first. Then run `mujoco.mj_forward` so the
    position stage is fresh, and read `data.xpos` for the body.

    `data.xpos` is documented in MuJoCo's own header as the "Cartesian position of body
    frame". That is NOT the same as `data.xipos`, the "Cartesian position of body com" —
    section 3 measures how far apart they are on this hand.

    Return a copy, not the live view.

    Example (at the home pose the arm hangs straight down):
        >>> model, data = load_arm()
        >>> body_world_position(model, data, "hand_right", HOME).round(3)
        array([ 0.  , -0.19,  0.85])

    Returns:
        np.ndarray of shape (3,), independent of `data`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public check — run it as often as you like.
def _check_body_world_position() -> None:
    model, data = load_arm()
    got = body_world_position(model, data, HAND_BODY, HOME)
    assert isinstance(got, np.ndarray) and got.shape == (3,), (
        f"expected a length-3 numpy array, got {type(got).__name__} with shape "
        f"{getattr(got, 'shape', None)} — return the row for this body, not the whole array."
    )
    reference = np.array([0.0, -0.19, 0.85])
    assert np.allclose(got, reference, atol=1e-9), (
        f"at the home pose this body's frame origin is {reference}, you returned {got}. If "
        "you are 0.05 m low you read data.xipos (the body's centre of mass) instead of "
        "data.xpos; if nothing moved when qpos changed, you did not call mujoco.mj_forward."
    )
    moved = body_world_position(model, data, HAND_BODY, READY)
    assert not np.allclose(moved, got), (
        "the ready pose gave the same answer as the home pose — you are not writing `qpos` "
        "into data before refreshing, so both calls measured the same arm."
    )
    first = body_world_position(model, data, HAND_BODY, HOME)
    body_world_position(model, data, HAND_BODY, READY)      # a later call must not disturb it
    assert np.allclose(first, np.array([0.0, -0.19, 0.85])), (
        f"the answer from an earlier call changed to {first} once a later call ran — you "
        "returned the live data.xpos view instead of a copy of it."
    )
    print(f"exercise 1 looks right: hand frame origin {got} at home, {moved} at ready")

In [ ]:
_try("exercise 1", _check_body_world_position)

## 3. Three points on one rigid body, and why it matters

The hand is a single rigid body. It still has at least three positions worth naming: the
frame origin its joints hang off, the centre of mass its inertia acts at, and the palm site
you actually want to put on a doorknob. Run this — the numbers are centimetres apart, and
centimetres is plenty to be wrong by.

In [ ]:
_m2, _d2 = load_arm()
_hid = mujoco.mj_name2id(_m2, mujoco.mjtObj.mjOBJ_BODY, HAND_BODY)
_sid2 = mujoco.mj_name2id(_m2, mujoco.mjtObj.mjOBJ_SITE, PALM_SITE)
_d2.qpos[:] = READY
mujoco.mj_forward(_m2, _d2)
print(f"data.xpos      (body frame origin)  {_d2.xpos[_hid]}")
print(f"data.xipos     (body centre of mass){_d2.xipos[_hid]}")
print(f"data.site_xpos (the palm site)      {_d2.site_xpos[_sid2]}")
print(f"\norigin to com : {np.linalg.norm(_d2.xipos[_hid] - _d2.xpos[_hid]) * 100:.1f} cm")
print(f"origin to palm: {np.linalg.norm(_d2.site_xpos[_sid2] - _d2.xpos[_hid]) * 100:.1f} cm")
print("one body, one set of joint axes, three different points — remember that in section 5")

## 4. Exercise 2 — `site_world_position`

Same shape as exercise 1, different array. A site is a massless frame welded to a body:
MuJoCo's modeling chapter describes an element defined inside a body as "fixed to the local
frame of that body and always moves with it". It takes part in no dynamics — it exists so
you can name a point on a robot without inventing a body for it.

<details><summary>💡 Hint 1 — what to think about</summary>

The palm is not a body, and sites have their own numbering. If your answer at home is the
hand body's frame origin, you looked the name up in the wrong id space or read the wrong
array. Two welded points also stay the same distance apart in every pose, and the check
measures exactly that against your exercise 1 — so finish that one first.

</details>

<details><summary>💡 Hint 2 — the approach</summary>

Start from your exercise 1 and change two things only: the object type you look the name up
under, and the array you read the row from. The optional `qpos` write, the forward pass and
the copy stay exactly as they were.

</details>

In [ ]:
def site_world_position(model, data, site_name: str, qpos=None) -> np.ndarray:
    """Return the world position of a named site, as a length-3 array.

    Identical in structure to `body_world_position`, with two substitutions: look the name up
    under `mujoco.mjtObj.mjOBJ_SITE`, and read `data.site_xpos`.

    This is the forward-kinematics map the rest of the lesson differentiates: it takes a
    `qpos` and returns a point in the world. Everything from here on calls it.

    Example:
        >>> model, data = load_arm()
        >>> site_world_position(model, data, "palm", HOME).round(3)
        array([ 0.   , -0.19 ,  0.745])

    Returns:
        np.ndarray of shape (3,), independent of `data`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_site_world_position() -> None:
    model, data = load_arm()
    got = site_world_position(model, data, PALM_SITE, HOME)
    assert isinstance(got, np.ndarray) and got.shape == (3,), (
        f"expected a length-3 numpy array, got {type(got).__name__}."
    )
    reference = np.array([0.0, -0.19, 0.745])
    assert np.allclose(got, reference, atol=1e-9), (
        f"at the home pose the palm sits at {reference}, you returned {got}. The value "
        f"{np.array([0.0, -0.19, 0.85])} is the HAND BODY's origin — read data.site_xpos, "
        "indexed by the SITE id, not data.xpos indexed by a body id."
    )
    hand = body_world_position(model, data, HAND_BODY, READY)
    palm = site_world_position(model, data, PALM_SITE, READY)
    gap = float(np.linalg.norm(palm - hand))
    assert abs(gap - 0.105) < 1e-9, (
        f"the palm should sit exactly 0.105 m from the hand frame origin in any pose, because "
        f"they are welded together; your two functions put them {gap:.4f} m apart."
    )
    print(f"exercise 2 looks right: palm {got} at home, and {gap * 100:.1f} cm from the hand "
          "frame origin in every pose")

In [ ]:
_try("exercise 2", _check_site_world_position, needs=("exercise 1",))

## 5. What a Jacobian column actually means

The Jacobian is the derivative of the map you just built. Column *i* answers exactly one
question: **if dof *i* alone moved at 1 rad/s, which way and how fast would this point
travel?** Stack those answers and you have a 3-by-nv matrix that turns joint velocity into
Cartesian velocity: `x_dot = J @ qvel`.

Two consequences worth holding on to:

- The columns are indexed by **degree of freedom**, not by joint and not by `qpos` slot. On
  this model those three happen to coincide, because every joint is a hinge. On a humanoid
  with a free-floating base they do not, and section 9 shows that happening.
- A dof that cannot move the point gets a column of **exactly** zero — not nearly zero.

Run this to see MuJoCo's own Jacobian for the palm, before you build one.

In [ ]:
_m3, _d3 = load_arm()
_sid3 = mujoco.mj_name2id(_m3, mujoco.mjtObj.mjOBJ_SITE, PALM_SITE)
_d3.qpos[:] = READY
mujoco.mj_forward(_m3, _d3)
_J = np.zeros((3, _m3.nv))
mujoco.mj_jacSite(_m3, _d3, _J, None, _sid3)
print("MuJoCo's translational Jacobian of the palm, at the ready pose (3 x nv):")
print(_J)
print(f"\nshape {_J.shape} = 3 world axes by nv={_m3.nv} degrees of freedom")
print(f"left-arm columns {LEFT_ARM_DOFS} are exactly zero: "
      f"{np.array_equal(_J[:, LEFT_ARM_DOFS], np.zeros((3, 2)))}")
print("the left arm cannot move the right palm, and the derivative knows it")

## 6. Exercise 3 — `analytic_jacobian`

`mujoco.mj_jac(model, data, jacp, jacr, point, body)` is the general form: it computes the
Jacobian of a **point** that is rigidly attached to a **body**. MuJoCo's API reference says
the convenience variants `mj_jacBody`, `mj_jacBodyCom` and `mj_jacSite` "call mj_jac
internally, with the center of the body, geom or site" — so they are one function evaluated
at three different points, which is why section 3's three points matter.

You will call the general form, with the palm's position as the point and the palm's own
body as the body. Pass `None` for `jacr`: you want translation only.

<details><summary>💡 Hint 1 — what to think about</summary>

`mj_jac` is a C function with an output argument: it fills an array you allocate and hands
back nothing. How many columns does that array need — one per `qpos` slot, or one per degree
of freedom? On this arm the two agree only because every joint is a hinge. And which stage
must have run for the per-dof motion axes to exist at all?

</details>

<details><summary>💡 Hint 2 — the approach</summary>

Follow the docstring's five steps in order. The two that bite: the forward pass happens
inside your function every time, even when the caller has just run one; and the body you
pass is the one the site is welded to, read from the model, with the site's world position
as the point. A site id reused as a body id names a different body. Return the array you
filled.

</details>

In [ ]:
def analytic_jacobian(model, data, site_name: str, qpos=None) -> np.ndarray:
    """Return the 3-by-nv translational Jacobian of a named site, from `mujoco.mj_jac`.

    In order:

    1. If `qpos` is given, write it into `data.qpos`.
    2. `mujoco.mj_forward(model, data)`. This is not optional and it is not merely about
       accuracy: the Jacobian is built from `data.cdof`, the per-dof motion axes, which the
       position stage produces. Ask for a Jacobian without it and you get zeros. The cell
       below this one shows that happening.
    3. Find the site id, then the body it is attached to: `int(model.site_bodyid[site_id])`.
    4. Allocate `jacp = np.zeros((3, model.nv))` — `mj_jac` writes INTO the array you hand it
       and returns None, so a `J = mujoco.mj_jac(...)` gives you None every time.
    5. Call `mujoco.mj_jac(model, data, jacp, None, point, body_id)` where `point` is the
       site's world position as a float array, and return `jacp`.

    Example:
        >>> model, data = load_arm()
        >>> J = analytic_jacobian(model, data, PALM_SITE, READY)
        >>> J.shape
        (3, 10)
        >>> bool(np.array_equal(J[:, LEFT_ARM_DOFS], np.zeros((3, 2))))
        True

    Returns:
        np.ndarray of shape (3, model.nv).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_analytic_jacobian() -> None:
    model, data = load_arm()
    got = analytic_jacobian(model, data, PALM_SITE, READY)
    assert got is not None, (
        "you returned None — mujoco.mj_jac writes into the array you pass it and returns "
        "nothing, so return the array you allocated, not the call's result."
    )
    assert got.shape == (3, model.nv), (
        f"shape is {got.shape}, expected (3, {model.nv}) — 3 world axes by nv dofs. A "
        f"(3, {model.nq}) answer means you sized it off nq, which only agrees here by luck."
    )
    reference = np.zeros((3, model.nv))
    mujoco.mj_jacSite(model, data, reference, None,
                      mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, PALM_SITE))
    assert np.abs(got - reference).max() < 1e-12, (
        f"your Jacobian differs from mujoco.mj_jacSite by up to "
        f"{np.abs(got - reference).max():.3e}. They are the same computation at the same "
        "point, so they should agree to the last bit: check that your `point` is the site's "
        "world position and your `body` is the site's body."
    )
    assert np.abs(got).max() > 1e-9, (
        "your Jacobian is all zeros. That is what mj_jac returns when the position stage has "
        "not run — call mujoco.mj_forward before you ask for it."
    )
    assert np.array_equal(got[:, LEFT_ARM_DOFS], np.zeros((3, 2))), (
        f"the left-arm columns came out as {got[:, LEFT_ARM_DOFS]}, and they must be exactly "
        "zero: those two joints cannot move the right palm."
    )
    stale = mujoco.MjData(model)          # never had mj_forward called on it at all
    again = analytic_jacobian(model, stale, PALM_SITE, READY)
    assert np.abs(again - reference).max() < 1e-12, (
        "given a data object that has never been through the pipeline your function gave a "
        "different answer — it must refresh the position stage itself, not rely on the "
        "caller having done it."
    )
    print(f"exercise 3 looks right: matches mj_jacSite to {np.abs(got - reference).max():.1e}, "
          f"and dofs {LEFT_ARM_DOFS} are exactly zero")

In [ ]:
# The trap, measured. A fresh mjData, kinematics ONLY, then ask for a Jacobian.
_m4, _d4 = load_arm()
_sid4 = mujoco.mj_name2id(_m4, mujoco.mjtObj.mjOBJ_SITE, PALM_SITE)
_d4.qpos[:] = READY
mujoco.mj_kinematics(_m4, _d4)          # the position of things: computed
_J_partial = np.zeros((3, _m4.nv))
mujoco.mj_jacSite(_m4, _d4, _J_partial, None, _sid4)
print(f"after mj_kinematics alone: site position {_d4.site_xpos[_sid4]}  <- already correct")
print(f"                           Jacobian, largest entry {np.abs(_J_partial).max():.3e}")
mujoco.mj_comPos(_m4, _d4)              # the motion axes of each dof: now computed
_J_full = np.zeros((3, _m4.nv))
mujoco.mj_jacSite(_m4, _d4, _J_full, None, _sid4)
print(f"after mj_comPos as well  : Jacobian, largest entry {np.abs(_J_full).max():.3e}")
print("verdict:", "one derived quantity was ready and the other was not"
      if np.abs(_J_partial).max() == 0.0 and np.abs(_J_full).max() > 0.0
      else "unexpected — read the two numbers above before trusting the next section")

In [ ]:
_try("exercise 3", _check_analytic_jacobian)

## 7. Exercise 4 — the one that settles it: finite differences

This is the heart of the lesson. `mj_jac` hands you a matrix; nothing so far proves it is
the derivative of the map you built in exercise 2. There is exactly one way to find out
without trusting anybody: perturb one dof, watch the palm move, divide.

Use **central** differences — perturb up *and* down — because the error of
`(f(q+h) - f(q-h)) / 2h` falls off as h² while the one-sided version falls off as h. You
will measure that difference in section 8 rather than take it on faith.

Difference only the dofs in `dofs`. On this model that is a convenience; the two left-arm
columns are zero anyway. On a 27-dof humanoid it is the difference between a check you run
after every change and one you run once and stop bothering with.

<details><summary>💡 Hint 1 — what to think about</summary>

Every evaluation of your forward map writes into `data.qpos`. So where does your base pose
live while you work? If it is the live `data.qpos` — or the caller's own array — it walks
away from you one dof at a time. And how far apart are your two samples: one step, or two?
An error about the size of the step itself is the signature of a one-sided difference.

</details>

<details><summary>💡 Hint 2 — the approach</summary>

Take one private copy of the base pose before the loop. For each dof you were given, build
an up-copy and a down-copy that differ from it by `eps` in that slot only, evaluate
`site_world_position` at both, and divide their difference by the full distance between
the two samples. Leave every other column at zero. After the loop, evaluate once more at the
base pose so `data` ends where it began.

</details>

In [ ]:
def finite_difference_jacobian(model, data, site_name: str, qpos=None,
                               dofs=ARM_DOFS, eps: float = FD_EPS) -> np.ndarray:
    """Approximate the site's 3-by-nv translational Jacobian by central differences.

    Start from a base pose: `qpos` if it was given, otherwise whatever `data.qpos` currently
    holds. Take a copy of it — you are about to perturb it repeatedly, and mutating the
    caller's array is how this function acquires a memory.

    For each dof `i` in `dofs`:

        q_plus  = base.copy();  q_plus[i]  += eps
        q_minus = base.copy();  q_minus[i] -= eps
        column i = (site_world_position(..., q_plus) - site_world_position(..., q_minus))
                   / (2 * eps)

    Columns for dofs you were not asked about stay zero. When you are done, leave `data` at
    the base pose, so a caller who passed you their working `data` gets it back unperturbed.

    WHY `q[i] += eps` IS LEGITIMATE HERE, AND USUALLY IS NOT: it perturbs dof `i` only
    because every joint in this model is a hinge, so `nq == nv` and qpos slot `i` is dof `i`.
    A free or ball joint stores orientation as a four-number quaternion against a three-number
    angular velocity, so the slots stop lining up and adding `eps` to one of them is not
    "move dof i" at all — it is "make this quaternion slightly not a unit quaternion".
    Section 9 loads such a model and prints its nq and nv. MuJoCo's own `mj_integratePos` is
    the general tool for this.

    Example:
        >>> model, data = load_arm()
        >>> Jfd = finite_difference_jacobian(model, data, PALM_SITE, READY)
        >>> Jfd.shape
        (3, 10)

    Returns:
        np.ndarray of shape (3, model.nv), zero in every column not listed in `dofs`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_finite_difference_jacobian() -> None:
    model, data = load_arm()
    analytic = analytic_jacobian(model, data, PALM_SITE, READY)
    numeric = finite_difference_jacobian(model, data, PALM_SITE, READY)
    assert numeric.shape == (3, model.nv), (
        f"shape is {numeric.shape}, expected (3, {model.nv}) — build the full-width matrix "
        "and leave the columns you were not asked about at zero."
    )
    worst = float(np.abs(numeric - analytic).max())
    assert worst < JAC_TOL, (
        f"the numeric and analytic Jacobians disagree by up to {worst:.3e}, and the tolerance "
        f"is {JAC_TOL:.0e}. If the error is around {FD_EPS:.0e} you built a one-sided "
        "difference; if it is huge, check that you divided by 2*eps and not by eps."
    )
    base = READY.copy()
    finite_difference_jacobian(model, data, PALM_SITE, base)
    assert np.array_equal(base, READY), (
        "your function modified the qpos array it was handed — perturb a copy."
    )
    drift = float(np.abs(np.asarray(data.qpos) - READY).max())
    assert drift < 1e-12, (
        f"data.qpos came back {drift:.2e} away from the base pose — you left it at the last "
        "perturbation. Put it back after the loop; a caller's data must come back the way it "
        "went in."
    )
    data.qpos[:] = READY
    mujoco.mj_forward(model, data)
    implicit = finite_difference_jacobian(model, data, PALM_SITE)
    assert (np.abs(implicit - analytic).max() < 1e-8
            and float(np.abs(np.asarray(data.qpos) - READY).max()) < 1e-12), (
        "called with no qpos — differencing about the pose data already holds — either the "
        "answer or the restored pose was wrong. Take a COPY of data.qpos as your base: keep "
        "the live array and every evaluation you run writes into it, so the base walks out "
        "from under you one dof at a time."
    )
    only_two = finite_difference_jacobian(model, data, PALM_SITE, READY, dofs=[5, 6])
    assert np.array_equal(only_two[:, [0, 1, 2, 3, 4, 7, 8, 9]], np.zeros((3, 8))), (
        "asked for dofs [5, 6] you filled in other columns as well — difference only the "
        "dofs you were given."
    )
    assert np.abs(only_two[:, [5, 6]] - analytic[:, [5, 6]]).max() < JAC_TOL, (
        "the two columns you were asked for do not match the analytic ones."
    )
    print(f"exercise 4 looks right: the numeric and analytic Jacobians of the palm agree to "
          f"{worst:.2e} at eps={FD_EPS:.0e}, well inside the {JAC_TOL:.0e} tolerance")

In [ ]:
_try("exercise 4", _check_finite_difference_jacobian, needs=("exercise 2", "exercise 3"))

## 8. How small should the step be? Measure it

Two errors fight each other. **Truncation** error is the curvature the difference quotient
ignores; it shrinks as the step shrinks, like h² for central differences. **Round-off**
error comes from subtracting two nearly-equal doubles and dividing by a tiny number; it
*grows* as the step shrinks, like machine epsilon over h.

So the total error is a U. Find its bottom with your own function.

In [ ]:
def step_size_sweep(steps=(1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9, 1e-10)):
    """Print the agreement between your numeric and analytic Jacobians at each step size."""
    model, data = load_arm()
    analytic = analytic_jacobian(model, data, PALM_SITE, READY)
    rows = []
    print(f"  {'eps':>10s}  {'max |numeric - analytic|':>26s}")
    for eps in steps:
        err = float(np.abs(finite_difference_jacobian(model, data, PALM_SITE, READY, eps=eps)
                           - analytic).max())
        rows.append((eps, err))
        print(f"  {eps:>10.0e}  {err:>26.3e}")
    best_eps, best_err = min(rows, key=lambda r: r[1])
    print(f"\n  best step on this machine: eps={best_eps:.0e}, error {best_err:.2e}")
    assert best_eps not in (steps[0], steps[-1]), (
        f"the best step was {best_eps:.0e}, at one end of the sweep — the U has no bottom "
        "inside the range you swept, so widen it before drawing a conclusion."
    )
    big = dict(rows)[1e-2] if 1e-2 in dict(rows) else None
    if big is not None:
        print(f"  a 1e-2 step is {big / best_err:,.0f}x worse (curvature you stepped over);")
    print(f"  a 1e-10 step is {dict(rows)[1e-10] / best_err:,.0f}x worse "
          "(two nearly-equal doubles, subtracted).")
    print("  This is why the tolerance in this lesson is a stated number and not 'about right'.")
    return rows


_try("step-size sweep", step_size_sweep, needs=("exercise 3", "exercise 4"))

## 9. Why `qpos[i] += eps` is a privilege, not a technique

Everything above leans on `nq == nv`. MuJoCo's computation chapter states that nq exceeds
nv whenever quaternions represent 3D orientations. Here is a four-line model with one ball
joint, so you can watch the two numbers come apart instead of taking the sentence on trust.

In [ ]:
_BALL_MJCF = """
<mujoco>
  <worldbody><body name="b" pos="0 0 1">
    <joint name="ball" type="ball"/><geom type="sphere" size="0.05"/>
  </body></worldbody>
</mujoco>
"""
_ball = mujoco.MjModel.from_xml_string(_BALL_MJCF)
print(f"this lesson's arm : nq={MODEL.nq:2d}  nv={MODEL.nv:2d}  -> qpos slot i IS dof i")
print(f"one ball joint    : nq={_ball.nq:2d}  nv={_ball.nv:2d}  -> "
      f"{_ball.nq - _ball.nv} extra position slot, and the mapping is gone")
print("On that model, adding eps to a qpos slot does not move a dof; it makes a quaternion")
print("slightly non-unit. mujoco.mj_integratePos is the general tool. Your arm is the easy case.")

## 10. Exercise 5 — inverse kinematics by damped least squares

Forward is easy: angles in, position out. Backwards is the hard direction, and the honest
way is iterative. At the current pose, ask the Jacobian for the joint motion that would
move the palm along the error vector, take that step, and look again.

The naive step is the pseudo-inverse, `dq = J⁺ e`. It explodes near a singularity — a pose
where some Cartesian direction has become unreachable at any joint velocity — because it
cheerfully divides by a singular value approaching zero. The fix, published independently
by Wampler and by Nakamura and Hanafusa in 1986, is to solve a damped system instead:

```
dq = Jᵀ (J Jᵀ + λ² I)⁻¹ e
```

λ trades accuracy for stability. Section 11 measures what it buys, on a pose where the
undamped version does not merely struggle: it fails outright.

<details><summary>💡 Hint 1 — what to think about</summary>

Four things decide whether this solver is honest: which way the error points (the palm must
move *toward* the target), whether λ enters squared, whether every step is clipped to the
joint limits, and what you report when you run out of iterations. Read the `damping` and
`tol` you were handed, not the module's constants, and start `history` afresh on each call.

</details>

<details><summary>💡 Hint 2 — the approach</summary>

Copy `q_start` into a working pose. Each iteration: place the palm at the working pose,
record the residual, and return straight away if it is under `tol` — the loop index is then
the number of steps taken. Otherwise take the Jacobian's columns for `dofs`, solve the damped
3-by-3 system for the step, add it and clip. If the loop runs out, measure the residual once
more at the pose you finished in before reporting it, with `converged` False.

</details>

In [ ]:
IK_DAMPING = 0.05
IK_TOL = 1e-4
IK_MAX_ITERS = 120

# Two targets in world coordinates. The first is inside the arm's reach; the second is a
# metre from the shoulder, and the arm is 0.635 m long. Neither number is asserted anywhere —
# your solver reports what it finds.
REACHABLE_TARGET = np.array([0.45, -0.10, 1.15])
UNREACHABLE_TARGET = np.array([0.5963, -0.4881, 0.6346])


def ik_damped_least_squares(model, data, site_name: str, target, q_start=HOME,
                            dofs=ARM_DOFS, damping: float = IK_DAMPING,
                            tol: float = IK_TOL, max_iters: int = IK_MAX_ITERS) -> dict:
    """Drive a site onto a Cartesian target by damped least squares. Report what happened.

    Start from `q_start` (copy it — do not write into the caller's array). Then, up to
    `max_iters` times:

    1. `x = site_world_position(model, data, site_name, q)` and `e = target - x`.
    2. `residual = float(np.linalg.norm(e))`; append it to `history`.
    3. If `residual < tol`, stop: you have converged. Report the iteration count as the
       number of steps you actually TOOK, so a solve that was already on target reports 0.
    4. `J = analytic_jacobian(model, data, site_name, q)[:, dofs]` — a 3-by-len(dofs) block.
    5. `dq = J.T @ np.linalg.solve(J @ J.T + damping ** 2 * np.eye(3), e)`. Use
       `np.linalg.solve` on the 3-by-3 system rather than forming an inverse.
    6. `q[dofs] = np.clip(q[dofs] + dq, JOINT_LO[dofs], JOINT_HI[dofs])`. The clip matters:
       these joints have travel limits, and a solver that ignores them reports a pose the
       robot cannot adopt.

    If you run out of iterations, report the residual AT THE POSE YOU FINISHED IN — measure
    it again, do not reuse the last one you computed — and `converged=False`.

    Example:
        >>> model, data = load_arm()
        >>> out = ik_damped_least_squares(model, data, PALM_SITE, REACHABLE_TARGET)
        >>> out["converged"], bool(out["residual"] < IK_TOL)
        (True, True)

    Returns:
        dict with exactly these five keys:
          "qpos"        np.ndarray (nq,), the pose you finished in
          "residual"    float, metres between the site and the target at that pose
          "iterations"  int, steps taken
          "converged"   bool, True only if the residual fell below `tol`
          "history"     list of float, the residual at the start of each iteration
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_ik() -> None:
    model, data = load_arm()
    out = ik_damped_least_squares(model, data, PALM_SITE, REACHABLE_TARGET)
    assert set(out) == {"qpos", "residual", "iterations", "converged", "history"}, (
        f"keys were {sorted(out)} — return a dict with exactly those five names."
    )
    assert out["converged"], (
        f"the reachable target was missed: residual {out['residual']:.4f} m after "
        f"{out['iterations']} iterations. If the residual grew, check the sign of e — it is "
        "target minus current position, not the other way round."
    )
    assert out["residual"] < IK_TOL, (
        f"converged is True but the residual is {out['residual']:.3e}, above the {IK_TOL:.0e} "
        "tolerance — report the residual at the pose you actually finished in."
    )
    reached = site_world_position(model, data, PALM_SITE, out["qpos"])
    assert np.linalg.norm(reached - REACHABLE_TARGET) < IK_TOL, (
        f"the qpos you returned puts the palm at {reached}, which is not within tolerance of "
        f"{REACHABLE_TARGET} — the reported residual and the returned pose disagree."
    )
    assert len(out["history"]) == out["iterations"] + 1, (
        f"history has {len(out['history'])} entries after {out['iterations']} iterations; "
        "append the residual at the START of every iteration, including the one that "
        "discovers you have converged."
    )
    assert np.all(out["qpos"][ARM_DOFS] >= JOINT_LO[ARM_DOFS] - 1e-12) and \
           np.all(out["qpos"][ARM_DOFS] <= JOINT_HI[ARM_DOFS] + 1e-12), (
        f"the pose you returned is outside the joint limits: {out['qpos'][ARM_DOFS]} against "
        f"lo {JOINT_LO[ARM_DOFS]} hi {JOINT_HI[ARM_DOFS]} — clip after every step."
    )
    far = ik_damped_least_squares(model, data, PALM_SITE, UNREACHABLE_TARGET)
    assert not far["converged"], (
        f"the solver reported convergence on a target a metre from a 0.635 m arm, with "
        f"residual {far['residual']:.4f}."
    )
    assert far["iterations"] == IK_MAX_ITERS, (
        f"it gave up after {far['iterations']} iterations instead of using all "
        f"{IK_MAX_ITERS} — only the tolerance test may end the loop early."
    )
    assert far["residual"] > 0.1, (
        f"an unreachable target should leave a large residual; you report "
        f"{far['residual']:.4f} m."
    )
    print(f"exercise 5 looks right:")
    print(f"  reachable target   : converged in {out['iterations']} iterations, residual "
          f"{out['residual']:.2e} m")
    print(f"  unreachable target : gave up after {far['iterations']}, residual "
          f"{far['residual']:.4f} m — honestly reported, not hidden")

In [ ]:
_try("exercise 5", _check_ik, needs=("exercise 2", "exercise 3"))

## 11. What the damping is actually for

At the home pose the arm hangs straight down, fully extended. It cannot move the palm along
its own axis at any joint velocity, so the Jacobian loses rank: one of its singular values
is not small, it is zero. Every solve in section 10 started there.

Run this. The undamped normal equations do not degrade gracefully; NumPy refuses them.

In [ ]:
_m5, _d5 = load_arm()
_sid5 = mujoco.mj_name2id(_m5, mujoco.mjtObj.mjOBJ_SITE, PALM_SITE)


def _palm_jac(qpos):
    _d5.qpos[:] = qpos
    mujoco.mj_forward(_m5, _d5)
    out = np.zeros((3, _m5.nv))
    mujoco.mj_jacSite(_m5, _d5, out, None, _sid5)
    return out[:, ARM_DOFS]


_e = np.array([0.0, 0.0, -0.02])          # ask the palm to move 2 cm further down
print(f"  {'elbow (rad)':>12s} {'smallest singular value':>24s} "
      f"{'|dq| at lambda=0':>18s} {'|dq| at lambda=0.05':>20s}")
for _elbow in (0.0, -1e-4, -1e-3, -1e-2, -0.1, -0.5):
    _q = np.zeros(MODEL.nq)
    _q[5] = _elbow
    _Jk = _palm_jac(_q)
    _sv = np.linalg.svd(_Jk, compute_uv=False)[-1]
    _cells = []
    for _lam in (0.0, 0.05):
        try:
            _dq = _Jk.T @ np.linalg.solve(_Jk @ _Jk.T + _lam ** 2 * np.eye(3), _e)
            _cells.append(f"{np.linalg.norm(_dq):.3f}")
        except np.linalg.LinAlgError as _exc:
            _cells.append(f"{type(_exc).__name__}")
    print(f"  {_elbow:>12.4f} {_sv:>24.2e} {_cells[0]:>18s} {_cells[1]:>20s}")
print("\n  Read the top two rows. Exactly at full extension the undamped solve is refused")
print("  outright; a ten-thousandth of a radian away it is accepted, and asks for a joint")
print("  velocity no arm can deliver. The damped column never notices either event.")

In [ ]:
def damping_sweep(values=(0.001, 0.01, 0.05, 0.2, 0.5, 1.0)):
    """Solve the reachable target at several damping values and print the trade-off."""
    model, data = load_arm()
    print(f"  {'lambda':>8s} {'iterations':>11s} {'residual (m)':>14s} {'converged':>10s}")
    rows = []
    for lam in values:
        out = ik_damped_least_squares(model, data, PALM_SITE, REACHABLE_TARGET, damping=lam)
        rows.append((lam, out))
        print(f"  {lam:>8.3f} {out['iterations']:>11d} {out['residual']:>14.3e} "
              f"{str(out['converged']):>10s}")
    fastest = min((r for r in rows if r[1]["converged"]), key=lambda r: r[1]["iterations"])
    print(f"\n  fewest iterations among the ones that converged: lambda={fastest[0]}")
    print("  Small damping is fast when the pose is healthy and fragile when it is not;")
    print("  large damping is slow and unbothered. That is the whole trade.")
    return rows


_try("damping sweep", damping_sweep, needs=("exercise 5",))

## 12. Common mistakes

- **Writing `qpos` and reading a derived array straight away.** `site_xpos`, `xpos`, `cdof`
  and friends are outputs of the position stage. Until `mj_forward` runs, you are reading
  the previous pose — and it looks perfectly plausible.
- **Asking for a Jacobian after `mj_kinematics` alone.** You get zeros, silently. The
  motion axes come from `mj_comPos`. Section 6 measures it.
- **`data.xpos` when you wanted `data.xipos`, or either when you wanted `site_xpos`.** One
  rigid hand, three different points, and on this model they are centimetres apart.
- **Returning the live view.** `data.xpos[i]` is a window into `mjData`, not a value. The
  next forward pass rewrites the answer you thought you had saved.
- **`J = mujoco.mj_jac(...)`.** It returns None and writes into the array you passed. So
  does every variant.
- **Sizing the Jacobian by `nq`.** It has `nv` columns. They coincide on this model and
  will not on a humanoid with a floating base.
- **One-sided differences.** `(f(q+h) - f(q)) / h` has error of order h, not h². At
  h = 1e-6 that is a million times worse, and it will still look approximately right.
- **Chasing a smaller step size.** Below the bottom of the U you are differencing round-off.
  Section 8 shows the error climbing again at 1e-10.
- **Using the pseudo-inverse near a singularity.** It does not warn you. It either raises,
  or asks for hundreds of radians per second, which on hardware is a very loud noise.
- **Not clipping to the joint limits.** A solver that ignores them converges beautifully to
  a pose the robot cannot adopt.
- **Reporting the last residual you computed instead of the one at the pose you returned.**
  They differ by exactly one step, which is the step that was supposed to help.

In [ ]:
# Two of those mistakes, made on purpose, so you can see what they look like unlabelled.
# Neither number below is typed; both are measured.
_m6, _d6 = load_arm()
_sid6 = mujoco.mj_name2id(_m6, mujoco.mjtObj.mjOBJ_SITE, PALM_SITE)
_d6.qpos[:] = READY
mujoco.mj_forward(_m6, _d6)
_kept = _d6.site_xpos[_sid6]                 # the live view
_snapshot = np.array(_d6.site_xpos[_sid6])   # a copy of it
_d6.qpos[:] = HOME
mujoco.mj_forward(_m6, _d6)                  # the arm moves; only one of the two follows it
print("saved the palm position, then moved the arm home:")
print(f"  the copy you took : {_snapshot}")
print(f"  the view you kept : {_kept}   <- silently rewritten under you")
_J6 = np.zeros((3, _m6.nv))
print(f"  mujoco.mj_jacSite returned: {mujoco.mj_jacSite(_m6, _d6, _J6, None, _sid6)}"
      f"   <- and the answer, norm {np.linalg.norm(_J6):.3f}, went into the array you passed")

## 13. Self-check

1. You set `data.qpos[3] = 0.5` and immediately read `data.site_xpos[palm]`. It is
   unchanged. Why?
   - (a) `qpos` is read-only while a model is compiled
   - (b) that joint does not move the palm
   - (c) `site_xpos` is an output of the position stage, and nothing has recomputed it since
         you wrote to `qpos`
   - (d) sites do not move, only bodies do

2. Your `mj_jacSite` call returns a 3-by-nv matrix of zeros, but `data.site_xpos` is
   correct. What is the most likely cause?
   - (a) the site is attached to the world body
   - (b) `mj_kinematics` ran but `mj_comPos` did not, so the per-dof motion axes in
         `data.cdof` were never computed
   - (c) the model has no actuators, so nothing can move
   - (d) the Jacobian must be requested before forward kinematics, not after

3. You compare your finite-difference Jacobian against the analytic one at eps = 1e-2 and
   again at eps = 1e-10. Both disagree badly. What is going on?
   - (a) the analytic Jacobian is wrong; MuJoCo cannot be trusted for this
   - (b) both step sizes are too large
   - (c) the two are failing for opposite reasons — at 1e-2 the difference quotient steps
         over the curvature of the map, and at 1e-10 it is dividing round-off by a tiny
         number; the total error is a U with its bottom in between
   - (d) central differences are only valid for linear functions

4. Your inverse-kinematics solver runs fine for weeks, then one day asks for a joint
   velocity of several hundred radians per second. What happened, and what fixes it?
   - (a) the target moved too fast; slow the target down
   - (b) the pose reached a singularity where the Jacobian lost rank, so the undamped step
         divided by a near-zero singular value; adding the λ² term bounds the step instead
   - (c) the model's joint limits were set too wide
   - (d) floating-point error accumulated in `qpos` over many iterations

Answers, with reasoning, are in this lesson's worked solution in the course repository.

In [ ]:
# Put your four letters here and run the cell. It marks them without revealing the answer:
# a wrong letter sends you back to the section that measured it, which is the point.
SELF_CHECK = {1: "?", 2: "?", 3: "?", 4: "?"}

_ANSWER_DIGESTS = {1: "404dc788be3dd0cc", 2: "393a5382c69dee3c",
                   3: "cdf2dd3fa9437e78", 4: "bbed6f0b6d3476f1"}
_ANSWER_SECTIONS = {
    1: "section 1 — the stale reading you printed before calling mj_forward",
    2: "section 6 — the cell that ran mj_kinematics alone and printed the largest entry",
    3: "section 8 — the step-size sweep you ran, and which end of it climbed",
    4: "section 11 — the table of singular values and undamped step sizes you measured",
}


def _check_self_check(answers: dict = None) -> None:
    """Mark the four multiple-choice answers in SELF_CHECK, naming where to look again."""
    answers = SELF_CHECK if answers is None else answers
    wrong = []
    for q, digest in sorted(_ANSWER_DIGESTS.items()):
        got = str(answers.get(q, "?")).strip().lower()
        if hashlib.sha256(f"F15-L02-q{q}-{got}".encode()).hexdigest()[:16] != digest:
            wrong.append(q)
    for q in sorted(_ANSWER_DIGESTS):
        note = f"  -> re-read {_ANSWER_SECTIONS[q]}" if q in wrong else ""
        print(f"  q{q}: {'wrong' if q in wrong else 'right'}{note}")
    assert not wrong, (
        f"questions {wrong} are still wrong. Each one names the section that answers it "
        "above — go back to the measurement you ran there rather than guessing a letter."
    )
    print("self-check: all four right")


def _check_self_check_answered() -> None:
    """The board's view of the self-check: letters left at "?" are not started, not wrong."""
    if all(str(v).strip() == "?" for v in SELF_CHECK.values()):
        raise NotImplementedError("not answered yet — put your four letters in SELF_CHECK "
                                  "above, then re-run this cell.")
    _check_self_check()


_try("self-check", _check_self_check_answered)

## What you built, and where it goes next

A forward-kinematics map you can call on any pose, its derivative from MuJoCo, and — the
part that makes the other two trustworthy — a numerical check that the second really is the
derivative of the first, to a tolerance you chose from a sweep you ran.

The Jacobian is the hinge of the whole flagship. F15-L03 uses its transpose to turn a
Cartesian force into joint torques; the gait and balance lessons use the centre-of-mass
Jacobian the same way; and any time a controller of yours behaves strangely, the
finite-difference check you wrote here is how you find out whether the derivative or the
controller is lying.

In [ ]:
# Your progress board. Every check runs again here, quietly, so the board describes your code
# as it stands now: an exercise that was waiting on another you have since finished is marked
# afresh. In a script or under CI a failed check still ends the run non-zero; in a notebook it
# is a printed line.
_BOARD = [("exercise 1", _check_body_world_position, ()),
          ("exercise 2", _check_site_world_position, ("exercise 1",)),
          ("exercise 3", _check_analytic_jacobian, ()),
          ("exercise 4", _check_finite_difference_jacobian, ("exercise 2", "exercise 3")),
          ("exercise 5", _check_ik, ("exercise 2", "exercise 3")),
          ("self-check", _check_self_check_answered, ())]


def _progress_board() -> list:
    """Re-run every check without its output, print one line per exercise, return failures."""
    with contextlib.redirect_stdout(io.StringIO()):
        for label, check, needs in _BOARD:
            _try(label, check, needs)
    marks = {"passed": "✅", "failed": "❌", "not started": "⏳"}
    width = max(len(name) for name in _EXERCISES.values())
    print("progress board")
    for label, _, needs in _BOARD:
        state = _STATUS.get(label, "not started")
        waiting = [n for n in needs if _STATUS.get(n) != "passed"]
        note = ("  (waiting on " + " and ".join(waiting) + ")" if state == "not started"
                and waiting else "  (re-run its cell for the hint)" if state == "failed" else "")
        print(f"  {marks[state]} {label:<11s} {_EXERCISES[label]:<{width}s} {state}{note}")
    done = sum(_STATUS.get(label) == "passed" for label, _, _ in _BOARD)
    print(f"{done} of {len(_BOARD)} complete")
    return [label for label, state in _STATUS.items() if state == "failed"]


if __name__ == "__main__":
    _failed = _progress_board()
    if _failed:
        _message = "checks failed: " + ", ".join(_failed)
        if "ipykernel" in sys.modules:
            print(f"\n{_message} — each one printed its hint in its own cell above.")
        else:
            raise SystemExit(_message)